# Backbone Rigid Invariant (BRI) Examples

This notebook introduces **`backbone-rigid-invariant`**, a Python package that describes
the 3D shape of a protein backbone using numbers that **do not depend on how the
protein is oriented in space**.

## What are BRI and LAI?

Imagine sitting on the Cα atom of residue *i*, looking towards the N and C atoms.
**BRI (Backbone Rigid Invariant)** records the positions (coordinates) of the backbone atoms
$(N, Cα, C)_{i+1}$ of the *next* residue in this local viewpoint (coordinate system). Because you are always "sitting on the previous residue", coordinates of the atoms describe the **relative** geometry between neighbours and don't change when the whole protein is rotated or moved. This description is **invariant** under rigid motion.

- **BRI**: 9 numbers per residue, describing the coordinates of atoms N, Cα, and C of the next residue in a coordinate system of the current residue; measured in Armstrong.
- **LAI (Length-Angle Invariant)**: 3 bond lengths, 3 bond angles, and 3 torsion angles that have direct biological meaning.

## Table of Contents

1. [Loading a protein structure](#loading) — from the PDB, a local file, or a URL
2. [Computing BRI and LAI](#computing) — turning coordinates into invariants
3. [Saving results](#saving) — exporting to CSV for use in other tools
4. [Visualisation](#plotting) — making diagrams and barcodes to visualise the invariants
5. [Quick Reference](#quickref)


## Setup

Run the cell below to import everything we need. If you see an error about a missing
package, uncomment and run the installation line first.

In [ ]:
import os
os.makedirs('tutorial_output', exist_ok=True)

# Standard scientific Python libraries
import matplotlib.pyplot as plt

# The BRI package
import bri
from bri import ProteinChain, ProteinEntry

print(f"bri version: {bri.__version__}")

## 1. Loading a Protein Structure <a id="loading"></a>

The main class you will use is **`ProteinEntry`**. It represents a single protein [entry](https://www.youtube.com/watch?v=jSk0KgcQWl8), which may contain a few entities and chains.
You can load one by providing one of the options below:
- a **4-letter PDB ID** (e.g. [`"1hho"`](https://www.rcsb.org/structure/1HHO)) — a `.cif` file be used to load the structure automatically from the
  [RCSB PDB](https://www.rcsb.org/),
- a **path to a local `.pdb`, `.cif` or `.bcif` file**, or
- a **URL** pointing to a CIF file.

In [ ]:
entry = ProteinEntry.from_cif("1hho")

print(f"PDB ID: {entry.pdb_id}")
print(f"Number of chains: {entry.num_chains}")
for c in entry.chains:
    print(f"  Chain {c.chain_id}: {c.num_residues} residues, "
          f"{'polypeptide' if c.polypeptide else 'other polymer'}")

# Pick a specific chain by its letter:
chain_a = entry["A"]
print(f"\nChain A has {chain_a.num_residues} residues")

`ProteinEntry` allows you pick any chains by assessing the properties of `ProteinChain`, such as `model_id`, `chain_id`, `entity_id` etc.

In [ ]:
# Select chain(s) by their properties — returns a list of all matches.
# Multiple filters combine by AND; a list value means "any of".
target_model_id = 1
print(f'Chains in {entry.pdb_id} with model_id=={target_model_id}:')
print(entry.get_chains(model_id=target_model_id, polypeptide=True))
print()

target_length = 145
print(f'Chains in {entry.pdb_id} with residue number < {target_length}:')
print(entry.get_chains(residues=lambda x: len(x) < target_length))

print(f'\nSome available attributes to assess: {list(entry.chains[0].__dict__.keys())[:-3]}')

### Loading one specific chain at once: `ProteinChain`

If you want to work with one specific chain in a PDB entry, use `ProteinChain`. It selects the chain specified from
the `ProteinEntry` by providing the details to identify the chain, e.g. `model_id`, `chain_id`. 

In [ ]:
# Load chain A, model 1 of 1HHO (an oxidoreductase from E. coli)
chain = ProteinChain.from_cif("1hho", model_id=1, chain_id="A")
 
print(f"PDB ID:       {chain.pdb_id}")
print(f"Chain:        {chain.chain_id}")
print(f"Model:        {chain.model_id}")
print(f"Entity ID:    {chain.entity_id}")
print(f"Entity Type:  {chain.entity_type}")
print(f"Polymer type: {'polypeptide' if chain.polypeptide else 'other'}")
print(f"Residues:     {chain.num_residues}")
print(f"Total atoms:  {chain.num_atoms}")

### Inspecting residues and atoms

A `ProteinChain` is organised as a list of **residues**, each containing its **atoms**.
You can loop over them, access individual atoms by name and get the position of each atom.

In [ ]:
# Look at the first 5 residues
for res in chain.residues[:5]:
    print(res)

In [ ]:
# Pick the first residue and access its backbone atoms
first_residue = chain.residues[0]

n_atom  = first_residue.n      # backbone nitrogen
ca_atom = first_residue.ca     # backbone C-alpha
c_atom  = first_residue.c      # backbone carbonyl carbon

print(f"Residue: {first_residue.name}  (position {first_residue.seq_id})")
print(f"  N   —  x={n_atom.x:7.3f}  y={n_atom.y:7.3f}  z={n_atom.z:7.3f}")
print(f"  Cα  —  x={ca_atom.x:7.3f}  y={ca_atom.y:7.3f}  z={ca_atom.z:7.3f}")
print(f"  C   —  x={c_atom.x:7.3f}  y={c_atom.y:7.3f}  z={c_atom.z:7.3f}")

### Checking BRI computability

BRI computation requires **complete** backbone atoms and confirmed coordinates, which means the invariant can be complete iff each residue has exact three backbone atoms (N, Cα, C).
On the example of PDB entry `6cks`, let's check what happens with a chain having incomplete residues.

In [ ]:
incomplete_chain = ProteinChain.from_cif('6cks', 1, 'A', check_clean_flag=True)
print(incomplete_chain)

print('Rows with missing values:')
inv = incomplete_chain.get_invariant()
inv[inv.isna().any(axis=1)]



---
## 2. Computing Invariants <a id="computing"></a>

Now that we have a chain loaded, we can compute its invariants. There are three
levels of detail you can ask for:

| What you get | How many numbers per residue | What they tell you |
|-------------|---------------------------|--------------------|
| **BRI** | 3 weak invariants + 9 complete invariants | The relative 3D positions of N, Cα, C |
| **LAI** | 9 | bond lengths, bond angles and torsion angles |

### 2.1 Computing BRI

Call `chain.get_invariant()` to get the BRI for every residue as a table (DataFrame).

In [ ]:
bri_df = chain.get_invariant()

print(f"Table size: {bri_df.shape[0]} residues × {bri_df.shape[1]} columns")
bri_df.head(10)

**What the columns mean:**

| Column | Biological meaning |
|--------|-------------------|
| `residue_id` | Position of the residue in the chain (label sequence number) |
| `residue_label` | One-letter amino acid code |
| `x(AN)`, `x(AC)`, `y(AC)` | Auxiliary projected coordinates (weak invariants) for reconstruction |
| `x(N)`, `y(N)`, `z(N)` | Position of the **next** residue's N atom, in the **current** residue's local frame |
| `x(A)`, `y(A)`, `z(A)` | Position of the next Cα (A = alpha carbon) |
| `x(C)`, `y(C)`, `z(C)` | Position of the next C |


> **Tip:** The first residue has no "previous"
> residue to use as a reference, so its values are computed against itself and 
> look different from the rest.

### 2.2 Computing BRI with LAI (bond angles & torsion angles)

Pass `invariant_type='lai'` to also get bond angles and torsion angles.
These are the **Length-Angle Invariants** and are more intuitive to interpret
because they correspond to familiar geometric quantities.

In [ ]:
bri_lai = chain.get_invariant('lai')

# Using 'lai' to compute bond lengths, bond angles, and torsion angles:
bri_lai.head(10)

**Angle columns explained:**

| Column | What it measures | Typical range |
|--------|-----------------|---------------|
| `length(N)` | Length of bond: Cᵢ₋₁–Nᵢ | ~1.336Å |
|`length(A)`| Length of bond: Nᵢ–Cαᵢ | ~1.459Å |
|`length(C)`| Length of bond: Cαᵢ–Cᵢ | ~1.525Å |
| `angle(N)` | Angle at N: Cᵢ₋₁–Nᵢ–Cαᵢ | ~120° |
| `angle(A)` | Angle at Cα: Nᵢ–Cαᵢ–Cᵢ | ~110° |
| `angle(C)` | Angle at C: Cαᵢ–Cᵢ–Nᵢ₊₁ | ~116° |
| `tau(NA)` | Torsion around Nᵢ–Cαᵢ bond | relates to φ (phi) |
| `tau(AC)` | Torsion around Cαᵢ–Cᵢ bond | relates to ψ (psi) |
| `tau(CN)` | Torsion around Cᵢ–Nᵢ₊₁ bond | relates to ω (omega) |

> **Connection to Ramachandran angles:** The torsion angles τ(NA), τ(AC), and τ(CN)
> are closely related to the familiar φ, ψ, and ω dihedral angles. They capture the
> same backbone degrees of freedom.

---
## 3. Saving Results to CSV <a id="saving"></a>

All results are tables (pandas DataFrames), so you can save them with `.to_csv()`.
This lets you open them in Excel, R, or any other analysis tool.

In [ ]:
# Save per-residue BRI

chain.save_invariant("tutorial_output/1hho_A_bri.csv", 'bri')
print("✓ Saved: tutorial_output/1hho_A_bri.csv")

Invariant of the whole entry can also be computed and saved in a single csv file:


In [ ]:
filename = f"tutorial_output/{entry.pdb_id}_bri.csv"

entry = ProteinEntry.from_cif("1hho")
entry.save_invariant(filename, 'bri')


---
## 4. Visualisation <a id="plotting"></a>

The package includes two built-in visualisation styles. We define them as functions
below so you can reuse them with any chain.

- **BID (Backbone Invariant Diagram)** — line plots showing how each invariant
  changes along the sequence.
- **BIB (Backbone Invariant Barcode)** — a colour-coded strip where each row
  corresponds to the N, Cα, or C atom and colour encodes the invariant values.

### 4.1 Backbone Invariant Diagram (BID)

The BID shows the 9 strong invariants as 9 separate line plots. Each row shows
one coordinate (x, y, or z) for one atom type (N, Cα, or C).

- **Red** lines = N atom
- **Green** lines = Cα atom
- **Blue** lines = C atom

In [ ]:
fig_bid = chain_a.generate_BID()
# fig_bid.savefig("tutorial_output/1hho_A_BID.png", dpi=150, bbox_inches="tight")


Each of the 9 panels is one coordinate of one backbone atom in the **local frame**.


### 4.2 Backbone Invariant Barcode (BIB)

The BIB shows the same information as a colour strip, where each of the 9 values
is mapped to red, green, or blue intensity. This makes it easy to spot patterns
across hundreds of residues at a glance.

In [ ]:
fig_bib = chain_a.generate_BIB()
# fig_bib.savefig("tutorial_output/1hho_A_BIB.png", dpi=150, bbox_inches="tight")

**How to read the BIB:**

- **Each row** represents the x, y, z coordinates of one backbone atom type (N, Cα, C),
  encoded as red, green, and blue colour channels.
- **Consistent colour blocks** = structurally regular regions.
- **Colour changes** = structural transitions.
- **White vertical stripes** = unusual or missing values.

---
## 5. Quick Reference <a id="quickref"></a>

### Loading structures

| You want to | Code |
|---------------|------|
| Load a single chain | `ProteinChain.from_cif("1abc", model_id=1, chain_id="A")` |
| Load all chains in an entry | `ProteinEntry.from_cif("1abc")` |
| Load from a local CIF file | `ProteinChain.from_cif("path/to/file.cif", model_id=1, chain_id="A")` |
| Load from a local PDB file | `ProteinEntry.from_pdb("path/to/file.pdb")` then `entry["A"]` |
| Check if a chain is protein | `chain.polypeptide` |
| Get residue count | `chain.num_residues` |
| Get atom count | `chain.num_atoms` |

### Computing invariants

| You want to | Code |
|---------------|------|
| Compute BRI | `chain.get_invariant('bri')` |
| Compute LAI | `chain.get_invariant('lai')` |

### Saving and plotting

| You want to | Code |
|---------------|------|
| Save to CSV | `chain.save_invarinat("output.csv")` |
| Draw a BID diagram | `chain.generate_BID()` |
| Draw a BIB barcode | `chain.generate_BIB()` |

---

<!-- ## Where to go next

- **Package documentation:** [https://backbone-rigid-invariant.readthedocs.io](https://backbone-rigid-invariant.readthedocs.io)
- **Source code & more examples:** [https://github.com/AAAAAkki/Backbone_Invariant](https://github.com/AAAAAkki/Backbone_Invariant)
- **Structure cleaning:** The package can also filter out incomplete residues, chain breaks,
  and clashes before computing invariants — see `bri.invariant_compare` and `bri.filter`
  modules for details.
- **Pairwise comparison:** Use `bri.group_invariant_compare()` to find similar chains
  by their BRI distances. -->